# Notebook 02: Self-Attention（自己注意機構）

Transformer の核心部分です。

**問い**：文「The bank was steep」の「bank」は川岸？銀行？
→ 周囲の単語（steep）を見れば川岸とわかる。

**Self-Attention** は「各トークンが他のトークンをどれだけ参照すべきか」を
実数の重みとして計算する仕組みです。

```
X (seq_len, d_model)
    ↓
Q = X @ W_Q,  K = X @ W_K,  V = X @ W_V
    ↓
Attention(Q, K, V) = softmax(Q @ K.T / √d_k) @ V
```

**使うライブラリ**：numpy のみ

In [1]:
import numpy as np
np.random.seed(0)
np.set_printoptions(precision=4, suppress=True)

# Notebook 01 と同じ設定で入力 X を再現
vocab_size = 8
d_model    = 4
seq_len    = 4

np.random.seed(42)
E = np.random.randn(vocab_size, d_model)

def positional_encoding(seq_len, d_model):
    PE = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            angle = pos / (10000 ** (2 * i / d_model))
            PE[pos, i]   = np.sin(angle)
            if i + 1 < d_model:
                PE[pos, i+1] = np.cos(angle)
    return PE

tokens = [2, 5, 1, 3]
PE = positional_encoding(seq_len, d_model)
X = E[tokens] + PE

print(f"入力 X: shape = {X.shape}")
print(X)

入力 X: shape = (4, 4)
[[-0.4695  1.5426 -0.4634  0.5343]
 [ 2.3071  0.3145  0.0676 -0.4247]
 [ 0.6751 -0.6503  1.5794  1.7674]
 [ 0.3831 -2.9033 -1.7246  0.4377]]


---
## 1. Q・K・V 行列の作り方

入力 X に 3 種類の重み行列をかけて **Query・Key・Value** を生成します：

$$Q = X W_Q, \quad K = X W_K, \quad V = X W_V$$

| 行列 | 役割 | 検索の比喩 |
|------|------|----------|
| **Q**（クエリ） | 「何を探したいか」 | 検索ワード |
| **K**（キー） | 「自分はどんな情報を持つか」 | 文書のタグ |
| **V**（バリュー） | 「参照されたときに渡す情報」 | 文書の内容 |

Q と K の内積が高い → そのペアに強く注意する → V を多く受け取る

In [2]:
np.random.seed(0)
d_k = d_model   # キー/クエリの次元（ここでは d_model と同じ）

# 学習される重み行列（ここではランダム初期化）
W_Q = np.random.randn(d_model, d_k) * 0.5
W_K = np.random.randn(d_model, d_k) * 0.5
W_V = np.random.randn(d_model, d_k) * 0.5

print(f"W_Q: shape = {W_Q.shape}")
print(W_Q)
print()

W_Q: shape = (4, 4)
[[ 0.882   0.2001  0.4894  1.1204]
 [ 0.9338 -0.4886  0.475  -0.0757]
 [-0.0516  0.2053  0.072   0.7271]
 [ 0.3805  0.0608  0.2219  0.1668]]



In [3]:
# Q, K, V を計算（行列積）
Q = X @ W_Q   # (seq_len, d_model) @ (d_model, d_k) = (seq_len, d_k)
K = X @ W_K
V = X @ W_V

print(f"Q: shape = {Q.shape}")
print(Q)
print()
print(f"K: shape = {K.shape}")
print(K)
print()
print(f"V: shape = {V.shape}")
print(V)

Q: shape = (4, 4)
[[ 1.2535 -0.9103  0.5882 -0.8906]
 [ 2.1635  0.296   1.1891  2.5395]
 [ 0.5793  0.8846  0.5275  2.249 ]
 [-2.1176  1.1679 -1.2188 -0.5321]]

K: shape = (4, 4)
[[-2.4362  1.2818  0.624  -0.2275]
 [ 1.0732 -0.4951  0.4657 -1.1886]
 [ 4.4814 -0.1318 -0.0023  0.1394]
 [ 2.3704  0.5876 -1.2004  1.1579]]

V: shape = (4, 4)
[[ 1.2641  1.6043 -0.1564 -0.5142]
 [-0.7579 -2.0509 -0.2539  0.0337]
 [-1.9781 -2.5681 -2.4461  2.3787]
 [-1.1633 -0.9962  1.6928 -1.0432]]


---
## 2. Scaled Dot-Product Attention（ステップごとの計算）

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

5 つのステップを 1 つずつ数値で確認します。

In [4]:
# Step 1: Q @ K^T — 各トークンペアの「類似度スコア」
# (seq_len, d_k) @ (d_k, seq_len) = (seq_len, seq_len)
scores_raw = Q @ K.T

print(f"Step 1: Q @ K^T  shape = {scores_raw.shape}")
print("（行 i, 列 j = トークン i がトークン j をどれだけ参照したいか）")
print()
print(scores_raw)

Step 1: Q @ K^T  shape = (4, 4)
（行 i, 列 j = トークン i がトークン j をどれだけ参照したいか）

[[-3.6511  3.1286  5.6121  0.6991]
 [-4.7274 -0.2892 10.0078  6.8156]
 [-0.46   -2.2438  2.7917  3.864 ]
 [ 6.0164 -2.7861 -9.7149 -3.4863]]


In [5]:
# Step 2: スケーリング — √d_k で割って勾配爆発を防ぐ
# d_k が大きいと内積の値が大きくなりすぎ, softmax が極端になる
scale = np.sqrt(d_k)
scores_scaled = scores_raw / scale

print(f"Step 2: スケーリング（÷ √{d_k} = ÷ {scale:.4f}）")
print()
print(scores_scaled)

Step 2: スケーリング（÷ √4 = ÷ 2.0000）

[[-1.8255  1.5643  2.8061  0.3496]
 [-2.3637 -0.1446  5.0039  3.4078]
 [-0.23   -1.1219  1.3959  1.932 ]
 [ 3.0082 -1.393  -4.8575 -1.7432]]


In [6]:
# Step 3: Causal Mask（因果マスク）
# GPT 系（デコーダ）では「未来のトークンを参照できない」制約が必要
# → 上三角をマイナス無限大にして softmax で 0 にする

# np.triu: 上三角行列（対角より上）。k=1 で対角を含まない上三角
causal_mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
print("Causal Mask（True の位置 = 参照禁止）:")
print(causal_mask.astype(int))
print()

scores_masked = scores_scaled.copy()
scores_masked[causal_mask] = -1e9   # 実質的に -∞

print("Step 3: マスク適用後スコア:")
print(scores_masked)

Causal Mask（True の位置 = 参照禁止）:
[[0 1 1 1]
 [0 0 1 1]
 [0 0 0 1]
 [0 0 0 0]]

Step 3: マスク適用後スコア:
[[-1.8255e+00 -1.0000e+09 -1.0000e+09 -1.0000e+09]
 [-2.3637e+00 -1.4462e-01 -1.0000e+09 -1.0000e+09]
 [-2.3000e-01 -1.1219e+00  1.3959e+00 -1.0000e+09]
 [ 3.0082e+00 -1.3930e+00 -4.8575e+00 -1.7432e+00]]


In [7]:
# Step 4: Softmax — スコアを確率（合計=1）に変換
def softmax(x, axis=-1):
    # max 引き算で数値安定化（オーバーフロー防止）
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

attn_weights = softmax(scores_masked, axis=-1)

print(f"Step 4: Attention 重み（各行の合計 = 1）:")
print(attn_weights)
print()
print("各行の合計:", attn_weights.sum(axis=-1))   # すべて 1.0 になる

Step 4: Attention 重み（各行の合計 = 1）:
[[1.     0.     0.     0.    ]
 [0.0981 0.9019 0.     0.    ]
 [0.154  0.0631 0.7828 0.    ]
 [0.9792 0.012  0.0004 0.0085]]

各行の合計: [1. 1. 1. 1.]


In [8]:
# Attention 重みの読み方
print("Attention 重みの解釈:")
for i in range(seq_len):
    attending = [(f"{attn_weights[i,j]:.3f}", f"tok{j}") for j in range(seq_len)]
    print(f"  トークン{i} → {attending}")
print()
print("→ 上三角（未来）は 0.0000 になっている（マスクの効果）")

Attention 重みの解釈:
  トークン0 → [('1.000', 'tok0'), ('0.000', 'tok1'), ('0.000', 'tok2'), ('0.000', 'tok3')]
  トークン1 → [('0.098', 'tok0'), ('0.902', 'tok1'), ('0.000', 'tok2'), ('0.000', 'tok3')]
  トークン2 → [('0.154', 'tok0'), ('0.063', 'tok1'), ('0.783', 'tok2'), ('0.000', 'tok3')]
  トークン3 → [('0.979', 'tok0'), ('0.012', 'tok1'), ('0.000', 'tok2'), ('0.008', 'tok3')]

→ 上三角（未来）は 0.0000 になっている（マスクの効果）


In [9]:
# Step 5: V との重み付き和 — 最終的な Attention 出力
# 各トークンの出力 = 参照したトークンの V の加重平均
attn_output = attn_weights @ V   # (seq_len, seq_len) @ (seq_len, d_k) = (seq_len, d_k)

print(f"Step 5: Attention 出力  shape = {attn_output.shape}")
print(attn_output)

Step 5: Attention 出力  shape = (4, 4)
[[ 1.2641  1.6043 -0.1564 -0.5142]
 [-0.5596 -1.6925 -0.2443 -0.0201]
 [-1.4017 -1.8928 -1.955   1.7851]
 [ 1.2181  1.5369 -0.1428 -0.511 ]]


---
## 3. 全 5 ステップのまとめ関数

In [10]:
def scaled_dot_product_attention(Q, K, V, mask=True):
    """Scaled Dot-Product Attention の完全実装"""
    seq_len, d_k = Q.shape
    
    # 1. スコア計算
    scores = Q @ K.T / np.sqrt(d_k)
    
    # 2. Causal Mask
    if mask:
        causal = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
        scores[causal] = -1e9
    
    # 3. Softmax → Attention 重み
    weights = softmax(scores, axis=-1)
    
    # 4. V との積
    return weights @ V, weights

output, weights = scaled_dot_product_attention(Q, K, V)
print("出力（上と同じになるか確認）:")
print(output)

出力（上と同じになるか確認）:
[[ 1.2641  1.6043 -0.1564 -0.5142]
 [-0.5596 -1.6925 -0.2443 -0.0201]
 [-1.4017 -1.8928 -1.955   1.7851]
 [ 1.2181  1.5369 -0.1428 -0.511 ]]


---
## 4. Multi-Head Attention（多頭注意機構）

1 つの Attention では 1 種類の「参照パターン」しか学べません。

**Multi-Head Attention** は `h` 個の独立した Attention を並列実行し、
それぞれが異なる側面（構文, 意味, 位置など）を学習できます。

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W_O$$

$$\text{head}_i = \text{Attention}(X W_{Q_i}, X W_{K_i}, X W_{V_i})$$

各ヘッドの次元は `d_k = d_model // num_heads`

In [11]:
num_heads = 2
d_k_head  = d_model // num_heads   # 各ヘッドの次元 = 4 // 2 = 2

np.random.seed(1)

# 各ヘッド用の重み行列（独立）
W_Qs = [np.random.randn(d_model, d_k_head) * 0.5 for _ in range(num_heads)]
W_Ks = [np.random.randn(d_model, d_k_head) * 0.5 for _ in range(num_heads)]
W_Vs = [np.random.randn(d_model, d_k_head) * 0.5 for _ in range(num_heads)]
W_O  = np.random.randn(d_model, d_model) * 0.5   # 連結後の線形変換

print(f"各ヘッドの次元 d_k_head = {d_k_head}")
print(f"W_Q[0] shape: {W_Qs[0].shape}  （d_model={d_model} → d_k_head={d_k_head}）")

各ヘッドの次元 d_k_head = 2
W_Q[0] shape: (4, 2)  （d_model=4 → d_k_head=2）


In [12]:
# 各ヘッドで独立に Attention を計算
heads = []
for h in range(num_heads):
    Q_h = X @ W_Qs[h]   # (seq_len, d_k_head)
    K_h = X @ W_Ks[h]
    V_h = X @ W_Vs[h]
    
    head_out, head_weights = scaled_dot_product_attention(Q_h, K_h, V_h)
    heads.append(head_out)
    
    print(f"Head {h+1} Attention 重み:")
    print(head_weights)
    print(f"Head {h+1} 出力: shape={head_out.shape}")
    print(head_out)
    print()

Head 1 Attention 重み:
[[1.     0.     0.     0.    ]
 [0.542  0.458  0.     0.    ]
 [0.3704 0.5817 0.0479 0.    ]
 [0.3933 0.0073 0.5975 0.0019]]
Head 1 出力: shape=(4, 2)
[[ 0.3459  0.3325]
 [-0.4027 -0.3358]
 [-0.5941 -0.5054]
 [ 0.4718  0.4566]]

Head 2 Attention 重み:
[[1.     0.     0.     0.    ]
 [0.4012 0.5988 0.     0.    ]
 [0.3135 0.4902 0.1964 0.    ]
 [0.0549 0.0089 0.4273 0.5089]]
Head 2 出力: shape=(4, 2)
[[-0.492   2.2224]
 [-0.4235  0.1578]
 [-0.2633  0.1947]
 [ 0.6672 -0.4973]]



In [13]:
# 全ヘッドを連結して最終出力を計算
concat = np.concatenate(heads, axis=-1)   # (seq_len, d_model)
mha_output = concat @ W_O                 # (seq_len, d_model)

print(f"連結後 concat: shape = {concat.shape}")
print(concat)
print()
print(f"Multi-Head Attention 出力 (concat @ W_O): shape = {mha_output.shape}")
print(mha_output)

連結後 concat: shape = (4, 4)
[[ 0.3459  0.3325 -0.492   2.2224]
 [-0.4027 -0.3358 -0.4235  0.1578]
 [-0.5941 -0.5054 -0.2633  0.1947]
 [ 0.4718  0.4566  0.6672 -0.4973]]

Multi-Head Attention 出力 (concat @ W_O): shape = (4, 4)
[[-1.2139  1.2118  0.5169 -0.5124]
 [-0.0696 -0.1639 -0.0454 -0.2385]
 [ 0.0691 -0.0957 -0.024  -0.1892]
 [ 0.235   0.0649 -0.0092  0.4202]]


In [14]:
# MultiHeadAttention クラスにまとめる
class MultiHeadAttention:
    def __init__(self, d_model, num_heads, seed=1):
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        rng = np.random.default_rng(seed)
        s = 0.5
        self.W_Qs = [rng.standard_normal((d_model, self.d_k)) * s for _ in range(num_heads)]
        self.W_Ks = [rng.standard_normal((d_model, self.d_k)) * s for _ in range(num_heads)]
        self.W_Vs = [rng.standard_normal((d_model, self.d_k)) * s for _ in range(num_heads)]
        self.W_O  = rng.standard_normal((d_model, d_model)) * s

    def forward(self, X):
        heads = []
        for h in range(self.num_heads):
            Q_h = X @ self.W_Qs[h]
            K_h = X @ self.W_Ks[h]
            V_h = X @ self.W_Vs[h]
            head_out, _ = scaled_dot_product_attention(Q_h, K_h, V_h)
            heads.append(head_out)
        return np.concatenate(heads, axis=-1) @ self.W_O

mha = MultiHeadAttention(d_model=4, num_heads=2)
out = mha.forward(X)
print(f"MultiHeadAttention.forward(X): shape = {out.shape}")
print(out)

MultiHeadAttention.forward(X): shape = (4, 4)
[[ 0.4865  0.1221  0.5062 -0.1845]
 [ 0.5434 -0.2771  0.9075 -0.3769]
 [-0.6828 -0.1698  1.1207  0.7527]
 [ 0.15   -0.1644  0.5605 -0.2357]]


---
## まとめ

| ステップ | 操作 | 形状の変化 |
|---------|------|----------|
| Q, K, V 生成 | `X @ W_Q/K/V` | `(seq, d_model)` → `(seq, d_k)` |
| スコア | `Q @ K.T / √d_k` | `(seq, d_k)` → `(seq, seq)` |
| Causal Mask | 上三角を -∞ | `(seq, seq)` |
| Softmax | 各行を確率化 | `(seq, seq)` |
| 重み付き和 | `weights @ V` | `(seq, seq)` → `(seq, d_k)` |
| 連結＋W_O | Multi-Head 版 | `(seq, d_model)` |

→ **次のノートブック `03_transformer_block.ipynb`** では
この MHA に Layer Norm と FFN を組み合わせて Transformer ブロックを完成させます。